# **Loading Packages**
_____

In [ ]:
import pandas as pd
import numpy as np

import requests
import time
from io import StringIO

from fredapi import Fred
import yfinance as yf

# **Calling Data**
______

## **FRED**

In [9]:
with open("API_key.txt", "r") as file:
    api_key = file.read().strip()
fred = Fred(api_key=api_key)

exchange_rate = fred.get_series('RBGBBIS')
oil_price = fred.get_series('DCOILBRENTEU')
bond_yield = fred.get_series('IRLTLT01GBM156N')
UK_GDP = fred.get_series('NGDPRSAXDCGBQ')

Fred_Data = pd.DataFrame({
    'ExchangeRate': exchange_rate,
    'OilPrice': oil_price,
    'BondYield': bond_yield,
    'rGDP': UK_GDP
})

## **Office of National Stats**

## **Yahoo Finance**

In [14]:
ftse = yf.Ticker("^FTSE")

ftse_data = ftse.history(
    start="1984-01-01",  
    end=pd.Timestamp.today().strftime('%Y-%m-%d'),
    interval="1d"  
)
ftse_quarterly = ftse_data.resample('QE').last() 

Thoughts on alternative resampling options:
- ftse_quarterly = ftse_data.resample('QE').mean()  
- ftse_quarterly = ftse_data.resample('QE').first()  
- ftse_quarterly = ftse_data['Close'].resample('QE').last()  

## **OECD**

In [32]:
def fetch_oecd(url, name):
    """Fetch OECD data and return cleaned DataFrame"""
    if 'format=' not in url:
        url += '&format=csvfilewithlabels'
    
    headers = {
        'Accept': 'application/vnd.sdmx.data+csv; charset=utf-8; labels=both',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    time.sleep(1)
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        
        # Clean: keep only essential columns
        df_clean = df[['TIME_PERIOD', 'OBS_VALUE']].copy()
        df_clean.columns = ['date', name]
        df_clean[name] = pd.to_numeric(df_clean[name], errors='coerce')
        return df_clean
    else:
        print(f"✗ {name}: Error {response.status_code}")
        return None

# ======================================================================
# Update URLs here
# ======================================================================
queries = {
    "house_prices": "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,/GBR.Q.RHP.?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "cpi": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_G20_PRICES@DF_G20_PRICES,/GBR.Q...PA...?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "unemployment": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,/GBR..._Z.Y._T.Y_GE15..Q?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions"
}

dfs = [fetch_oecd(url, name) for name, url in queries.items()]
OECDdata = pd.concat([df.set_index('date') for df in dfs if df is not None], axis=1).reset_index()

